Environment Setup

In [1]:

# Install required libraries
!pip install --upgrade transformers datasets torch

pip install --upgrade transformers tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from datasets import ClassLabel


Load dataset tokenizer & model

In [2]:

# Load dataset
print("Loading dataset...")
dataset = load_dataset("code_x_glue_cc_defect_detection")

# Load tokenizer and model
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:


# Tokenization function
def tokenize(batch):
    return tokenizer(batch["func"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(tokenize, batched=True)

# Rename 'target' column to 'labels' as expected by the Trainer
tokenized_dataset = tokenized_dataset.rename_column("target", "labels")

# Ensure labels are of type LongTensor for CrossEntropyLoss
# Commenting out previous attempt and enabling cast_column
# tokenized_dataset = tokenized_dataset.map(lambda examples: {'labels': torch.tensor(examples['labels'], dtype=torch.long)}, batched=True)
tokenized_dataset = tokenized_dataset.cast_column("labels", ClassLabel(num_classes=2))

# Set format for PyTorch
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=10,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
)

# Train the model
print("Starting training...")
trainer.train()

# Save the fine-tuned model
print("Saving model...")
model.save_pretrained("/content/drive/MyDrive/Assignments/AI Bug System_Project/bug_prediction_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Assignments/AI Bug System_Project/bug_prediction_model")

print("Model saved to /content/drive/MyDrive/Assignments/AI Bug System_Project/bug_prediction_model")

Tokenizing dataset...


Map:   0%|          | 0/21854 [00:00<?, ? examples/s]

Map:   0%|          | 0/2732 [00:00<?, ? examples/s]

Map:   0%|          | 0/2732 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/21854 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/Assignments/AI Bug System_Project/bug_prediction_model


In [ ]:
# @title
import json
from pathlib import Path

model_path = Path("/content/drive/MyDrive/Assignments/AI Bug System_Project/bug_prediction_model")
cfg_file = model_path / "tokenizer_config.json"

print("Files in folder:", sorted(p.name for p in model_path.iterdir()))

if cfg_file.exists():
    cfg = json.loads(cfg_file.read_text())
    print("tokenizer_config.json contents:", cfg)
    est = cfg.get("extra_special_tokens")
    print("extra_special_tokens raw:", est, type(est))
    # If it's a single string, convert to list
    if isinstance(est, str):
        cfg["extra_special_tokens"] = [est]
        cfg_file.write_text(json.dumps(cfg, indent=2))
        print("Converted extra_special_tokens to list and saved.")
else:
    print("tokenizer_config.json not found.")

Files in folder: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
tokenizer_config.json contents: {'add_prefix_space': False, 'backend': 'tokenizers', 'bos_token': '<s>', 'cls_token': '<s>', 'eos_token': '</s>', 'errors': 'replace', 'is_local': False, 'mask_token': '<mask>', 'model_max_length': 512, 'pad_token': '<pad>', 'sep_token': '</s>', 'tokenizer_class': 'RobertaTokenizer', 'trim_offsets': True, 'unk_token': '<unk>'}
extra_special_tokens raw: None <class 'NoneType'>


In [ ]:
# @title
import json
from pathlib import Path

cfg_path = Path("/content/drive/MyDrive/Assignments/AI Bug System_Project/bug_prediction_model/tokenizer_config.json")
cfg = json.loads(cfg_path.read_text())
cfg["extra_special_tokens"] = []
cfg_path.write_text(json.dumps(cfg, indent=2))
print("Set extra_special_tokens to an empty list.")


Set extra_special_tokens to an empty list.


In [3]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve and log in automatically
hf_token = userdata.get('hf_tok')
login(hf_token)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM

# -------------------------------
# Load the fine-tuned bug prediction model
# -------------------------------
model_path = "/content/drive/MyDrive/Assignments/AI Bug System_Project/bug_prediction_model"   # Updated to the correct save path
bug_tokenizer = AutoTokenizer.from_pretrained(model_path)
bug_model = AutoModelForSequenceClassification.from_pretrained(model_path)

# -------------------------------
# Load a bug-fix model (CodeT5)
# -------------------------------
fix_tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base", extra_special_tokens=[]) # Added extra_special_tokens=[]
fix_model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base")

# -------------------------------
# Example buggy code snippet
# -------------------------------
buggy_code = """
def divide(a, b)
    return a / b
print(divide(10, 10))
"""

# -------------------------------
# Bug Prediction
# -------------------------------
inputs = bug_tokenizer(buggy_code, return_tensors="pt", padding=True, truncation=True)
outputs = bug_model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1).item()
confidence = torch.softmax(outputs.logits, dim=1)[0][prediction].item()

print("Bug Prediction Result:")
if prediction == 1:
    print(f"⚠️ Bug detected with confidence {confidence:.2f}")
else:
    print(f"✅ No bug detected with confidence {confidence:.2f}")

# -------------------------------
# Bug Fix Suggestion
# -------------------------------
fix_inputs = fix_tokenizer(buggy_code, return_tensors="pt", padding=True, truncation=True)
fix_outputs = fix_model.generate(**fix_inputs, max_length=200)
fixed_code = fix_tokenizer.decode(fix_outputs[0], skip_special_tokens=True)

print("\nSuggested Fixed Code:")
print(fixed_code)


THE OUTPUT FOR ABOVE PIECE OF CODE.

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Bug Prediction Result:
⚠️ Bug detected with confidence 0.95

Suggested Fixed Code:
<extra_id_0>b / b
print(10)